# Huấn luyện mô hình DAN trên RAF-DB với Google Colab

Đây là Notebook huấn luyện mô hình **Distract Your Attention Network (DAN)** trên tập dữ liệu RAF-DB. Kiến trúc này tự động chú ý vào các vùng quan trọng trên khuôn mặt thay vì toàn bộ ảnh, từ đó cải thiện độ chính xác đáng kể.

## 1. Chuẩn bị Môi trường và Dữ liệu
Chạy ô dưới đây để Mount Google Drive và giải nén dữ liệu từ Drive vào máy ảo Colab để đọc dữ liệu siêu tốc.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# CHÚ Ý: Sửa đường dẫn bên dưới trỏ tới file RAF-DB.zip trên Drive của bạn
!unzip -q "/content/drive/MyDrive/DoAnCV/RAF-DB.zip" -d "/content/dataset"

## 2. Khởi tạo Mô hình DAN (Distract Your Attention Network)
Sử dụng backbone là ResNet18 kết hợp với các Attention Modules để phân loại.

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

class DAN(nn.Module):
    def __init__(self, num_class=7, num_head=4):
        super(DAN, self).__init__()
        # Sử dụng Backbone ResNet18 pre-trained
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.num_head = num_head
        
        # Module sinh Attention Map (vùng chú ý)
        self.conv_att = nn.Conv2d(512, self.num_head, kernel_size=1)
        
        # Classifier
        self.fc = nn.Linear(512, num_class)
        self.bn = nn.BatchNorm1d(num_class)

    def forward(self, x):
        x = self.features(x)
        
        att_map = self.conv_att(x)
        att_map = att_map.view(att_map.size(0), self.num_head, -1)
        att_map = F.softmax(att_map, dim=2)
        att_map = att_map.view(att_map.size(0), self.num_head, x.size(2), x.size(3))
        
        x_flat = x.view(x.size(0), 1, x.size(1), -1)
        att_flat = att_map.view(att_map.size(0), self.num_head, 1, -1)
        
        weighted_features = (x_flat * att_flat).sum(dim=-1)
        final_features = weighted_features.mean(dim=1)
        
        out = self.fc(final_features)
        out = self.bn(out)
        
        return out

# Kiểm tra nhanh
model = DAN()
print("Đã khởi tạo mô hình DAN thành công!")

## 3. Cấu hình Dữ liệu và Tăng cường (Augmentation)
Chống Overfitting (học vẹt) là rất quan trọng đối với tập nhỏ như RAF-DB.

In [ ]:
def get_dataloaders(data_dir, batch_size=32):
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=train_transform)
    test_dataset = datasets.ImageFolder(os.path.join(data_dir, 'test'), transform=test_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, test_loader, train_dataset.classes

## 4. Quá trình Huấn luyện
Quá trình Training sẽ tự động lưu lại model tốt nhất vào Google Drive (dựa trên Validation Accuracy).

In [ ]:
def train_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Bắt đầu huấn luyện trên thiết bị: {device}")

    # !!! QUAN TRỌNG: Sửa đường dẫn bên dưới tới thư mục đã giải nén !!!
    data_dir = '/content/dataset/RAF-DB'
    
    if not os.path.exists(data_dir):
        print(f"Không tìm thấy thư mục {data_dir}. Vui lòng kiểm tra lại đường dẫn!")
        return

    train_loader, test_loader, classes = get_dataloaders(data_dir, batch_size=32)
    print(f"Nhãn cảm xúc nhận diện: {classes}")

    model = DAN(num_class=len(classes), num_head=4).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
    
    # ĐÃ SỬA LỖI PYTORCH 2.2+: Bỏ tham số verbose=True
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    num_epochs = 30
    best_acc = 0.0

    for epoch in range(num_epochs):
        start_time = time.time()
        
        # --- PHASE 1: TRAIN ---
        model.train()
        running_loss, corrects, total = 0.0, 0, 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels.data)
            total += inputs.size(0)
            
        epoch_loss = running_loss / total
        epoch_acc = corrects.double() / total

        # --- PHASE 2: VALIDATION ---
        model.eval()
        val_corrects, val_total, val_running_loss = 0, 0, 0.0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_running_loss += loss.item() * inputs.size(0)
                
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                val_total += inputs.size(0)
                
        val_acc = val_corrects.double() / val_total
        val_loss = val_running_loss / val_total
        scheduler.step(val_acc)
        
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1:02d}/{num_epochs} | Time: {epoch_time:.0f}s | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            # Lưu trực tiếp vào Drive để không mất khi tắt máy
            torch.save(model.state_dict(), '/content/drive/MyDrive/DoAnCV/best_dan_model.pth')
            print("  -> [TỐT HƠN] Đã lưu model xịn nhất vào Google Drive!")

    print(f"Quá trình huấn luyện hoàn tất! Độ chính xác cao nhất đạt được: {best_acc:.4f}")

# Chạy lệnh train (Bỏ comment nếu sẵn sàng)
# train_model()

## 5. Đánh giá Mô hình (Confusion Matrix & Accuracy Per Class)
Sau khi train xong, bạn chạy ô này để load mô hình tốt nhất từ Google Drive và vẽ Confusion Matrix, đánh giá độ chính xác của từng cảm xúc một (Precision, Recall, F1-score).

In [ ]:
def evaluate_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Load lại dữ liệu test
    data_dir = '/content/dataset/RAF-DB'
    _, test_loader, classes = get_dataloaders(data_dir, batch_size=32)
    
    # Khởi tạo mô hình và tải trọng số tốt nhất đã lưu
    model = DAN(num_class=len(classes), num_head=4).to(device)
    model_path = '/content/drive/MyDrive/DoAnCV/best_dan_model.pth'
    if not os.path.exists(model_path):
        print(f"Không tìm thấy file {model_path}. Bạn đã train model chưa?")
        return
        
    model.load_state_dict(torch.load(model_path))
    model.eval()
    
    all_preds = []
    all_labels = []
    
    print("Đang đánh giá mô hình trên toàn bộ tập Test... (Có thể mất khoảng 10-20 giây)")
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    # Ánh xạ nhãn nếu RAF-DB đang dùng số (1-7) thay vì tên chữ.
    # Cấu trúc của RAF-DB: 1: Surprise, 2: Fear, 3: Disgust, 4: Happiness, 5: Sadness, 6: Anger, 7: Neutral
    label_mapping = {"1": "Surprise", "2": "Fear", "3": "Disgust", "4": "Happiness", "5": "Sadness", "6": "Anger", "7": "Neutral"}
    display_classes = [label_mapping.get(c, c) for c in classes]
    
    # 1. Báo cáo phân loại (Accuracy, Precision, Recall cho từng Class)
    print("\n================ BÁO CÁO ĐỘ CHÍNH XÁC =================")
    print(classification_report(all_labels, all_preds, target_names=display_classes))
    
    # 2. Vẽ Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=display_classes, yticklabels=display_classes, annot_kws={"size": 12})
    plt.title('Confusion Matrix - DAN Model trên RAF-DB', fontsize=15)
    plt.ylabel('Thực tế (True Label)', fontsize=12)
    plt.xlabel('Dự đoán (Predicted Label)', fontsize=12)
    plt.show()

# Bỏ comment dòng dưới để chạy sau khi quá trình Train ở Ô 4 đã hoàn tất
# evaluate_model()